# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de productos

Este notebook diagnostica el Bronze oficial `andinalog_productos.csv`. Conserva intactas sus siete columnas y todas las filas. No normaliza identificadores, corrige categorías, imputa valores ni elimina duplicados: el tratamiento corresponde al notebook 2 y requiere reglas aprobadas.

Cada ejecución genera cuatro archivos en `proyecto-integrador/andinalog_productos/notebook1/salidas/`: diagnosticado, problemas, cuarentena y reporte de calidad con SHA-256 del Bronze. Las salidas anteriores con esos nombres se reemplazan al ejecutar la última celda.


## 1 · Configuración y origen

En local, ejecuta el notebook desde una carpeta dentro de `practicasNotebookColab`. En Colab, selecciona `drive` o deja `auto` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `proyecto-integrador/`.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_productos.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-productos-diagnostico-v2"

COLUMNAS_ORIGINALES = [
    "producto_id", "nombre_producto", "categoria_logistica",
    "temperatura_conservacion_requerida_c", "tolerancia_temperatura_c",
    "precio_unitario_bob", "costo_unitario_bob",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if ((carpeta / "datasets" / CARPETA_DATASETS).is_dir()
                and (carpeta / "proyecto-integrador").is_dir()):
            return carpeta
    raise FileNotFoundError("Ejecuta el notebook dentro de practicasNotebookColab")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "proyecto-integrador" / "andinalog_productos" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_productos.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook1\salidas


## 2 · Carga y contrato

Se leen las siete columnas como texto y se conservan los vacíos. Las conversiones numéricas solo sirven para diagnosticar; no alteran el Bronze.


In [3]:

def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(
        ruta,
        dtype="string",
        encoding="utf-8-sig",
        keep_default_na=False
    )
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(
            f"Esquema inesperado. Faltantes: {faltantes}; "
            f"extras: {extras}; orden: {list(df.columns)}"
        )
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)

print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


Bronze: 63 filas × 7 columnas
SHA-256: 0ee6adc9905e0d41ec5a2646f64633cccbab53ac35ee66a95f9bd46687744eda


,producto_id,nombre_producto,categoria_logistica,temperatura_conservacion_requerida_c,tolerancia_temperatura_c,precio_unitario_bob,costo_unitario_bob
0,PROD-001,Producto Logistico 001,Fresco,4.0,2.0,23.7,20.26
1,PROD-002,Producto Logistico 002,Seco,20.0,5.0,101.83,82.37
2,PROD-003,Producto Logistico 003,Seco,20.0,5.0,83.08,68.01
3,PROD-004,Producto Logistico 004,Fresco,4.0,2.0,19.15,16.27
4,PROD-005,Producto Logistico 005,Seco,20.0,5.0,42.02,35.68


## 3 · Catálogo y reglas de diagnóstico

Se comprueban identificadores vacíos o mal formados, IDs repetidos y colisiones que aparecerían al quitar espacios y pasar a mayúsculas. También se revisan nombres vacíos, categorías fuera de `Fresco`, `Congelado` y `Seco`, campos numéricos faltantes o no convertibles, y tolerancia, precio o costo no positivos.

La colisión normalizada es solo una alerta: el notebook no cambia IDs ni decide qué fila conservar. No se imponen límites superiores de temperatura, precio o costo sin una regla de negocio aprobada.


In [4]:

CATALOGO_PROBLEMAS = pd.DataFrame([
    ("producto_id", "FALTANTE", "Identificador vacío"),
    ("producto_id", "FORMATO_INVALIDO", "No cumple PROD-### exactamente"),
    ("producto_id", "DUPLICADO", "Identificador repetido; se marca la aparición posterior"),
    ("producto_id", "COLISION_ID_NORMALIZADO", "Coincidencia tras normalizar espacios y mayusculas"),

    ("nombre_producto", "FALTANTE", "Nombre de producto vacío"),

    ("categoria_logistica", "FALTANTE", "Categoría logística vacía"),
    ("categoria_logistica", "VALOR_NO_RECONOCIDO",
     "Categoría fuera del dominio: Fresco, Congelado o Seco"),

    ("temperatura_conservacion_requerida_c", "FALTANTE",
     "Temperatura de conservación requerida vacía"),
    ("temperatura_conservacion_requerida_c", "NO_NUMERICO",
     "Temperatura no convertible a número"),

    ("tolerancia_temperatura_c", "FALTANTE", "Tolerancia vacía"),
    ("tolerancia_temperatura_c", "NO_NUMERICO",
     "Tolerancia no convertible a número"),
    ("tolerancia_temperatura_c", "NO_POSITIVA",
     "Tolerancia menor o igual que cero"),

    ("precio_unitario_bob", "FALTANTE", "Precio unitario vacío"),
    ("precio_unitario_bob", "NO_NUMERICO",
     "Precio unitario no convertible a número"),
    ("precio_unitario_bob", "NO_POSITIVO",
     "Precio unitario menor o igual que cero"),

    ("costo_unitario_bob", "FALTANTE", "Costo unitario vacío"),
    ("costo_unitario_bob", "NO_NUMERICO",
     "Costo unitario no convertible a número"),
    ("costo_unitario_bob", "NO_POSITIVO",
     "Costo unitario menor o igual que cero"),
], columns=["columna_afectada", "codigo_error", "criterio"])

display(CATALOGO_PROBLEMAS)

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = (
            df[columna] if columna in df.columns
            else pd.Series("", index=df.index, dtype="string")
        )
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def detectar_producto_id(df):
    original = df["producto_id"].astype("string")
    limpio = original.str.strip()
    normalizado = limpio.str.upper()
    colision = normalizado.duplicated(keep="first") & ~limpio.duplicated(keep="first")
    return [
        registrar_problema(
            df, limpio.eq(""),
            "producto_id", "FALTANTE", original
        ),
        registrar_problema(
            df,
            limpio.ne("") & ~original.str.fullmatch(r"PROD-\d{3}").fillna(False),
            "producto_id", "FORMATO_INVALIDO", original
        ),
        registrar_problema(
            df, limpio.duplicated(keep="first"),
            "producto_id", "DUPLICADO", original
        ),
        registrar_problema(
            df, colision, "producto_id", "COLISION_ID_NORMALIZADO", original
        ),
    ]

def detectar_nombre_producto(df):
    original = df["nombre_producto"].astype("string")
    limpio = original.str.strip()
    return [
        registrar_problema(
            df, limpio.eq(""),
            "nombre_producto", "FALTANTE", original
        )
    ]

def detectar_categoria(df):
    original = df["categoria_logistica"].astype("string")
    limpio = original.str.strip()
    permitidos = {"Fresco", "Congelado", "Seco"}
    return [
        registrar_problema(
            df, limpio.eq(""),
            "categoria_logistica", "FALTANTE", original
        ),
        registrar_problema(
            df,
            limpio.ne("") & ~original.isin(permitidos),
            "categoria_logistica", "VALOR_NO_RECONOCIDO", original
        ),
    ]

def detectar_numero(df, columna, exige_positivo=False):
    original = df[columna].astype("string")
    limpio = original.str.strip()
    numero = pd.to_numeric(limpio, errors="coerce")

    hallazgos = [
        registrar_problema(
            df, limpio.eq(""),
            columna, "FALTANTE", original
        ),
        registrar_problema(
            df, limpio.ne("") & numero.isna(),
            columna, "NO_NUMERICO", original
        ),
    ]

    if exige_positivo:
        codigo = "NO_POSITIVA" if columna == "tolerancia_temperatura_c" else "NO_POSITIVO"
        hallazgos.append(
            registrar_problema(
                df, numero.notna() & numero.le(0),
                columna, codigo, original
            )
        )
    return hallazgos

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))

    hallazgos = (
        detectar_producto_id(principal)
        + detectar_nombre_producto(principal)
        + detectar_categoria(principal)
        + detectar_numero(
            principal, "temperatura_conservacion_requerida_c",
            exige_positivo=False
        )
        + detectar_numero(
            principal, "tolerancia_temperatura_c",
            exige_positivo=True
        )
        + detectar_numero(
            principal, "precio_unitario_bob",
            exige_positivo=True
        )
        + detectar_numero(
            principal, "costo_unitario_bob",
            exige_positivo=True
        )
    )

    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(
        ["fila_bronze", "columna_afectada", "codigo_error"],
        kind="stable"
    ).reset_index(drop=True)

    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO

    columnas_por_fila = (
        problemas.groupby("fila_bronze")["columna_afectada"]
        .agg(lambda valores: "|".join(dict.fromkeys(valores)))
    )

    principal["columnas_con_problemas"] = (
        principal["fila_bronze"].map(columnas_por_fila).fillna("")
    )
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()

print(
    f"Principal: {len(df_diagnosticado):,}; "
    f"problemas: {len(df_problemas):,}; "
    f"filas en cuarentena: {len(df_cuarentena):,}"
)

resumen_problemas = (
    df_problemas
    .groupby(["columna_afectada", "codigo_error"])
    .size()
    .rename("filas")
    .reset_index()
)
display(resumen_problemas)
display(df_problemas)


,columna_afectada,codigo_error,criterio
0,producto_id,FALTANTE,Identificador vacío
1,producto_id,FORMATO_INVALIDO,No cumple PROD-### exactamente
2,producto_id,DUPLICADO,Identificador repetido; se marca la aparición ...
3,producto_id,COLISION_ID_NORMALIZADO,Coincidencia tras normalizar espacios y mayusc...
4,nombre_producto,FALTANTE,Nombre de producto vacío
5,categoria_logistica,FALTANTE,Categoría logística vacía
6,categoria_logistica,VALOR_NO_RECONOCIDO,"Categoría fuera del dominio: Fresco, Congelado..."
7,temperatura_conservacion_requerida_c,FALTANTE,Temperatura de conservación requerida vacía
8,temperatura_conservacion_requerida_c,NO_NUMERICO,Temperatura no convertible a número
9,tolerancia_temperatura_c,FALTANTE,Tolerancia vacía


Principal: 63; problemas: 14; filas en cuarentena: 13


,columna_afectada,codigo_error,filas
0,categoria_logistica,VALOR_NO_RECONOCIDO,3
1,producto_id,COLISION_ID_NORMALIZADO,1
2,producto_id,DUPLICADO,2
3,producto_id,FORMATO_INVALIDO,6
4,temperatura_conservacion_requerida_c,FALTANTE,2


,fila_bronze,columna_afectada,codigo_error,valor_original,version_diagnostico
0,8,producto_id,FORMATO_INVALIDO,prod-008,GIAD-M3-S4-productos-diagnostico-v2
1,16,producto_id,FORMATO_INVALIDO,prod-016,GIAD-M3-S4-productos-diagnostico-v2
2,18,temperatura_conservacion_requerida_c,FALTANTE,,GIAD-M3-S4-productos-diagnostico-v2
3,22,producto_id,FORMATO_INVALIDO,prod-022,GIAD-M3-S4-productos-diagnostico-v2
4,26,categoria_logistica,VALOR_NO_RECONOCIDO,conjelado,GIAD-M3-S4-productos-diagnostico-v2
5,28,categoria_logistica,VALOR_NO_RECONOCIDO,conjelado,GIAD-M3-S4-productos-diagnostico-v2
6,31,producto_id,FORMATO_INVALIDO,prod-031,GIAD-M3-S4-productos-diagnostico-v2
7,36,categoria_logistica,VALOR_NO_RECONOCIDO,conjelado,GIAD-M3-S4-productos-diagnostico-v2
8,46,producto_id,FORMATO_INVALIDO,prod-046,GIAD-M3-S4-productos-diagnostico-v2
9,49,temperatura_conservacion_requerida_c,FALTANTE,,GIAD-M3-S4-productos-diagnostico-v2


## 4 · Reporte y comprobaciones antes de exportar

El reporte identifica el CSV Bronze por SHA-256. Se comprueba que no se pierdan filas, que las siete columnas originales permanezcan iguales y que cada fila en cuarentena tenga un problema documentado.


In [5]:

def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()

    datos = [
        ("archivo_bronze", ruta.name),
        ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]

    datos += [
        (f"{col}:{codigo}", int(total))
        for (col, codigo), total in conteos.items()
    ]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(
    df_bronze, principal, problemas, cuarentena, reporte
):
    assert list(principal.columns) == [
        "fila_bronze",
        *COLUMNAS_ORIGINALES,
        "columnas_con_problemas",
        "en_cuarentena",
    ]

    pd.testing.assert_frame_equal(
        principal[COLUMNAS_ORIGINALES],
        df_bronze[COLUMNAS_ORIGINALES]
    )

    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()

    catalogo_pares = set(
        CATALOGO_PROBLEMAS[
            ["columna_afectada", "codigo_error"]
        ].apply(tuple, axis=1)
    )
    problemas_pares = set(
        problemas[
            ["columna_afectada", "codigo_error"]
        ].apply(tuple, axis=1)
    )
    assert problemas_pares.issubset(catalogo_pares)

    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(
    df_bronze,
    df_diagnosticado,
    df_problemas,
    RUTA_BRONZE,
    HASH_BRONZE
)

validar_resultados(
    df_bronze,
    df_diagnosticado,
    df_problemas,
    df_cuarentena,
    reporte_calidad
)

display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_productos.csv
1,sha256_bronze,0ee6adc9905e0d41ec5a2646f64633cccbab53ac35ee66...
2,version_diagnostico,GIAD-M3-S4-productos-diagnostico-v2
3,filas_bronze,63
4,filas_diagnosticadas,63
5,filas_en_cuarentena,13
6,filas_sin_cuarentena,50
7,problemas_detectados,14
8,categoria_logistica:VALOR_NO_RECONOCIDO,3
9,producto_id:COLISION_ID_NORMALIZADO,1


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

La última celda reemplaza las cuatro salidas después de verificar que el Bronze no cambió durante la ejecución. No sobrescribe el CSV original.


In [6]:

def exportar_salidas(
    directorio, tablas, ruta_bronze, huella_inicial
):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError(
            "El CSV Bronze cambió durante la ejecución; "
            "no se exportarán resultados"
        )

    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}

    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(
                mode="w",
                suffix=".csv",
                prefix=".tmp_productos_",
                dir=directorio,
                encoding="utf-8-sig",
                newline="",
                delete=False,
            ) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)

        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)

    return list(temporales)

tablas_salida = {
    "andinalog_productos_diagnosticado.csv": df_diagnosticado,
    "andinalog_productos_problemas.csv": df_problemas,
    "andinalog_productos_cuarentena.csv": df_cuarentena,
    "andinalog_productos_reporte_calidad.csv": reporte_calidad,
}

rutas_creadas = exportar_salidas(
    DIRECTORIO_SALIDAS,
    tablas_salida,
    RUTA_BRONZE,
    HASH_BRONZE
)

for ruta in rutas_creadas:
    print(ruta)

print("Bronze intacta; salidas anteriores reemplazadas")


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook1\salidas\andinalog_productos_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook1\salidas\andinalog_productos_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook1\salidas\andinalog_productos_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_productos\notebook1\salidas\andinalog_productos_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

Revisa los problemas y acuerda las reglas de tratamiento antes de generar el Silver de Productos. Las categorías, los IDs mal formados o repetidos y los valores faltantes permanecen como datos originales en este diagnóstico.
